# 1. CTC：从路径折叠到前缀搜索

**研究问题**：逐帧路径怎样对应文本？不提供帧级对齐时，如何合并所有合法路径的概率？为什么逐帧最大概率路径未必给出最大概率文本？

**方法**：使用小词表、三帧合成概率，依次实现 Exp001–003，并用穷举核对动态规划和前缀搜索。

```text
声学帧 [T,V] → CTC 逐帧概率
                    ├→ 路径 Collapse → token 序列
                    ├→ 前向/后向状态 [T,2U+1] → P(Y|X)
                    └→ 前缀束 {prefix: (p_blank,p_nonblank)} → N-best
```

**边界**：只研究单条合成序列；没有声学 Encoder、语音数据、训练或识别质量测量。对应平台题目：`asr_ctc_collapse`、`asr_ctc_forward`、`asr_ctc_prefix`。
**代码来源**：本仓库 `experiments/ctc_rnnt_aed/run.py`（基础版本提交 `e86b41b`）；Notebook 将其教学函数拆为逐格实验，不映射外部项目源码。


In [ ]:
import itertools
import math
from collections import defaultdict
import torch


## Exp001：CTC Collapse

**原理**：先合并原始路径中连续相同的 ID，再删除 blank。若先删 blank，`[1,0,1]` 会错误地折叠成一个 `1`。

**形状**：输入为长度 `T` 的整数路径；输出为长度 `U≤T` 的 token 元组。

In [ ]:
def ctc_collapse(path, blank=0):
    result, previous = [], None
    for token in path:
        if token != previous and token != blank:
            result.append(token)
        previous = token
    return tuple(result)


In [ ]:
assert ctc_collapse([1,1,0,1,2,2]) == (1,1,2)
assert ctc_collapse([0,0]) == ()
assert ctc_collapse([2,2,1,2], blank=2) == (1,)
print('带 blank 的重复 token：', ctc_collapse([1,1,0,1,2,2]))

## Exp002：CTC Forward-Backward

**原理**：将目标 `Y=(y₁,…,yᵤ)` 扩成 `(_,y₁,_,…,yᵤ,_)`，状态数为 `S=2U+1`。前向量 `α[t,s]` 汇总到达状态 `(t,s)` 的路径，后向量 `β[t,s]` 汇总从该状态到终点的剩余概率（不重复计当前帧）。运算在对数域使用 `logsumexp`。

**形状**：逐帧 `log_probs:[T,V]`；`alpha,beta:[T,S]`；序列 log 概率为标量。相同 token 之间不能直接跨过 blank 状态。

$$\log P(Y|X)=\operatorname{logsumexp}(\alpha[T-1,S-2],\alpha[T-1,S-1])$$

In [ ]:
def ctc_forward_backward(log_probs, target, blank=0):
    """Return log P(target), frame/state forward table and backward table."""
    t_max = log_probs.shape[0]
    extended = [blank]
    for token in target:
        extended += [token, blank]
    s_max = len(extended)
    neg = log_probs.new_tensor(float('-inf'))
    forward = [[neg for _ in extended] for _ in range(t_max)]
    forward[0][0] = log_probs[0, blank]
    if s_max > 1:
        forward[0][1] = log_probs[0, extended[1]]
    for t in range(1, t_max):
        for s, token in enumerate(extended):
            sources = [forward[t - 1][s]]
            if s:
                sources.append(forward[t - 1][s - 1])
            if s > 1 and token != blank and token != extended[s - 2]:
                sources.append(forward[t - 1][s - 2])
            forward[t][s] = torch.logsumexp(torch.stack(sources), 0) + log_probs[t, token]
    backward = [[neg for _ in extended] for _ in range(t_max)]
    backward[-1][-1] = log_probs.new_tensor(0.)
    if s_max > 1:
        backward[-1][-2] = log_probs.new_tensor(0.)
    for t in range(t_max - 2, -1, -1):
        for s in range(s_max):
            transitions = [log_probs[t + 1, extended[s]] + backward[t + 1][s]]
            if s + 1 < s_max:
                transitions.append(log_probs[t + 1, extended[s + 1]] + backward[t + 1][s + 1])
            if s + 2 < s_max and extended[s + 2] != blank and extended[s] != extended[s + 2]:
                transitions.append(log_probs[t + 1, extended[s + 2]] + backward[t + 1][s + 2])
            backward[t][s] = torch.logsumexp(torch.stack(transitions), 0)
    total = torch.logsumexp(torch.stack(forward[-1][-2:] if s_max > 1 else forward[-1]), 0)
    return total, torch.stack([torch.stack(row) for row in forward]), torch.stack([torch.stack(row) for row in backward])


In [ ]:
p = torch.tensor([[.2,.8],[.6,.4],[.3,.7]])
log_total, alpha, beta = ctc_forward_backward(p.log(), [1])
exact = sum(math.prod(p[t,k].item() for t,k in enumerate(path))
            for path in itertools.product(range(2), repeat=3)
            if ctc_collapse(path) == (1,))
assert abs(log_total.exp().item()-exact) < 1e-6
assert torch.allclose(torch.logsumexp(alpha[0,:2]+beta[0,:2],0), log_total)
print('逐帧概率', tuple(p.shape), '前/后向表', tuple(alpha.shape), 'P([1]|X)', log_total.exp().item())

## Exp003：CTC Prefix Beam Search

**原理**：一个文本前缀可由许多逐帧路径得到。为每个前缀保存“以 blank 结束”和“以非 blank 结束”两部分概率。遇到重复 token 时，非 blank 路径可留在原前缀；只有先前以 blank 结束的路径能扩展出第二个相同 token。

**形状**：输入 `[T,V]`，每帧剪枝后最多保留 `beam_size` 个前缀；输出为 `(token 元组, log 概率)` 列表。束宽有限时结果是近似搜索。

In [ ]:
def ctc_prefix_beam(log_probs, beam_size, blank=0):
    """Prefix masses are stored as (blank ending, nonblank ending)."""
    beams = {(): (1., 0.)}
    for frame in log_probs.exp().tolist():
        next_beams = defaultdict(lambda: [0., 0.])
        for prefix, (pb, pn) in beams.items():
            next_beams[prefix][0] += (pb + pn) * frame[blank]
            for token, probability in enumerate(frame):
                if token == blank:
                    continue
                if prefix and token == prefix[-1]:
                    next_beams[prefix][1] += pn * probability
                    next_beams[prefix + (token,)][1] += pb * probability
                else:
                    next_beams[prefix + (token,)][1] += (pb + pn) * probability
        order = sorted((p for p in next_beams if sum(next_beams[p]) > 0), key=lambda p: (-sum(next_beams[p]), p))[:beam_size]
        beams = {p: tuple(next_beams[p]) for p in order}
    return [(p, math.log(sum(beams[p]))) for p in sorted(beams, key=lambda p: (-sum(beams[p]), p))]


In [ ]:
p = torch.tensor([[.2,.5,.3],[.4,.4,.2],[.3,.2,.5]])
beams = ctc_prefix_beam(p.log(), beam_size=27)
reference = defaultdict(float)
for path in itertools.product(range(3), repeat=3):
    reference[ctc_collapse(path)] += math.prod(p[t,k].item() for t,k in enumerate(path))
assert set(dict(beams)) == set(reference)
assert all(abs(math.exp(score)-reference[prefix]) < 1e-6 for prefix,score in beams)
print('N-best 前三项：', beams[:3])

**验证依据与适用边界**：本页的路径穷举只对三帧小词表可行；长语音需要动态规划和剪枝。概率一致性不证明 CTC 模型已学到有效声学表示。下一本 Notebook 将使用候选分数讨论 AED 重打分。